# ハンズオン③ LangChain + ChromaDB で RAG パイプラインを構築する

**対応セクション**: 2-3「RAG (Retrieval-Augmented Generation)」  
**推奨所要時間**: 約 60 分  
**必要環境**: CPU のみで可（GPU 不要）

---

## このノートブックの目標

1. ドキュメントを読み込んでテキストチャンクに分割する
2. 埋め込みモデルでチャンクをベクトル化し ChromaDB に格納する
3. 質問に対して類似チャンクを検索し、LLM に渡して回答を生成する
4. チャンクサイズ（256 / 512 / 1024 token）を変えて検索精度を比較する
5. top-k 件数と回答品質の関係を実験する

## 0. セットアップ

In [ ]:
# 実行時間: 数秒
import os
import textwrap

os.environ.setdefault('HF_HOME', '/data/shared/hf_cache')
os.environ.setdefault('TRANSFORMERS_CACHE', '/data/shared/hf_cache')

print('セットアップ完了')

---

## Step 1: サンプルドキュメントの準備

実習用の技術文書をインラインで定義します。  
本番では `TextLoader` や `PyPDFLoader` で実際のファイルを読み込みます。

In [ ]:
# 実行時間: 数秒
# 実習用のサンプル技術文書
SAMPLE_DOCUMENTS = [
    {
        'title': '機械学習入門',
        'content': '''
機械学習とは、データからパターンを学習して予測や分類を行う人工知能の一分野です。
従来のプログラミングがルールを手動で記述するのに対し、機械学習はデータから自動的にルールを発見します。

機械学習の主要な手法は以下の3種類に分類されます。

1. 教師あり学習（Supervised Learning）
   入力と正解ラベルのペアを学習データとして使い、新しい入力に対して予測を行います。
   代表的なアルゴリズムには線形回帰、決定木、ニューラルネットワークがあります。
   用途: 分類（スパム検出）、回帰（価格予測）など。

2. 教師なし学習（Unsupervised Learning）
   正解ラベルなしにデータの構造やパターンを発見します。
   代表的なアルゴリズムにはk-meansクラスタリング、主成分分析（PCA）があります。
   用途: 顧客セグメンテーション、異常検出など。

3. 強化学習（Reinforcement Learning）
   エージェントが環境との相互作用を通じて報酬を最大化する行動方策を学習します。
   代表的なアルゴリズムにはQ学習、PPOがあります。
   用途: ゲームプレイ、ロボット制御など。
'''
    },
    {
        'title': '深層学習とニューラルネットワーク',
        'content': '''
深層学習（Deep Learning）は、多層のニューラルネットワークを使った機械学習の手法です。
「深層」とは、ネットワークが多くの隠れ層（hidden layer）を持つことを指します。

ニューラルネットワークの基本構造:
- 入力層（Input Layer）: データを受け取る
- 隠れ層（Hidden Layer）: 特徴を抽出・変換する
- 出力層（Output Layer）: 予測結果を出力する

活性化関数は非線形変換を導入し、複雑なパターン学習を可能にします。
代表的な活性化関数: ReLU（Rectified Linear Unit）、Sigmoid、Tanh。

学習は誤差逆伝播法（Backpropagation）によって行われます。
損失関数の勾配を計算し、連鎖律を使って各層の重みを更新します。

過学習（Overfitting）への対策:
- ドロップアウト（Dropout）: 学習中にランダムにニューロンを無効化
- L1/L2正則化: 重みの大きさにペナルティを与える
- バッチ正規化（Batch Normalization）: 各層の入力を正規化
- 早期終了（Early Stopping）: バリデーション損失が悪化したら学習を停止
'''
    },
    {
        'title': 'Transformer と大規模言語モデル',
        'content': '''
Transformer は 2017 年の論文「Attention is All You Need」で提案されたアーキテクチャです。
RNN や LSTM に代わり、自然言語処理の標準的なアーキテクチャとなりました。

Self-Attention の仕組み:
Self-Attention は、文中の各トークンが他のトークンとの関連度を計算します。
Query（Q）、Key（K）、Value（V）の3つの行列を使って注意スコアを計算します。
Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) * V

大規模言語モデル（LLM）の特徴:
- GPT系: デコーダのみのアーキテクチャ。次トークン予測で事前学習。
- BERT系: エンコーダのみのアーキテクチャ。マスクトークン予測で事前学習。
- LLaMA、Mistral等: オープンソースの GPT 系モデル。

ファインチューニング手法:
- SFT（教師ありファインチューニング）: 指示データで応答能力を付与
- RLHF / DPO: 人間のフィードバックで応答品質を改善
- LoRA: 低ランク行列で効率的にファインチューニング
'''
    },
]

# LangChain の Document 形式に変換
from langchain_core.documents import Document

documents = [
    Document(page_content=doc['content'], metadata={'title': doc['title']})
    for doc in SAMPLE_DOCUMENTS
]

print(f'ドキュメント数: {len(documents)}')
for d in documents:
    print(f'  - {d.metadata["title"]} ({len(d.page_content)} 文字)')

---

## Step 2: テキストのチャンキング

ドキュメントを検索しやすいサイズに分割します。  
**chunk_size を変えて精度がどう変わるか後で実験します。**

In [ ]:
# 実行時間: 数秒
from langchain.text_splitter import RecursiveCharacterTextSplitter

# ── ここを変えて実験しよう ──────────────────────────────────
CHUNK_SIZE    = 512   # 試す値: 256, 512, 1024
CHUNK_OVERLAP = 50
# ──────────────────────────────────────────────────────────

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    separators=['\n\n', '\n', '。', '、', ' ', ''],
)

chunks = splitter.split_documents(documents)

print(f'chunk_size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}')
print(f'チャンク数: {len(chunks)}')
print(f'\n--- チャンク例（最初の2件）---')
for i, chunk in enumerate(chunks[:2]):
    print(f'\n[チャンク {i+1}] ({len(chunk.page_content)} 文字)')
    print(chunk.page_content[:200])

---

## Step 3: エンベッディングと ChromaDB への格納

In [ ]:
# 実行時間: 約1〜2分（初回は埋め込みモデルのダウンロードあり）
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

print('埋め込みモデルを読み込み中...')
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True},
)

print('ChromaDB にチャンクを格納中...')
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=f'handson_chunk{CHUNK_SIZE}',
)

print(f'\nベクトルDB に {len(chunks)} 件のチャンクを格納しました')

---

## Step 4: 類似検索（Retrieval）

質問文をベクトル化し、コサイン類似度で上位 k 件のチャンクを取得します。

In [ ]:
# 実行時間: 数秒
# ── ここを変えて実験しよう ──────────────────────────────────
TOP_K = 3    # 試す値: 1, 3, 5
# ──────────────────────────────────────────────────────────

test_queries = [
    '過学習を防ぐ方法はありますか？',
    'Self-Attention の仕組みを教えてください',
    '教師なし学習の用途は？',
]

for query in test_queries:
    print(f'\n質問: {query}')
    results = vectorstore.similarity_search_with_score(query, k=TOP_K)
    for i, (doc, score) in enumerate(results):
        print(f'  [{i+1}] スコア={score:.3f} | {doc.page_content[:80].strip()}...')

---

## Step 5: RAG パイプライン全体の実行

検索したチャンクをコンテキストとして LLM に渡し、回答を生成します。  
ここでは軽量モデル（ELYZA または gpt2-ja）を使います。

In [ ]:
# 実行時間: 約1〜2分
from transformers import pipeline
import torch

# シンプルな RAG 関数（プロンプトを構築して回答）
def rag_answer(query: str, vectorstore, top_k: int = 3) -> str:
    # 1. 関連チャンクを検索
    docs = vectorstore.similarity_search(query, k=top_k)
    context = '\n\n'.join([doc.page_content for doc in docs])

    # 2. コンテキスト付きプロンプトを構築
    prompt = (
        f'以下の参考文書を使って質問に答えてください。\n\n'
        f'【参考文書】\n{context}\n\n'
        f'【質問】\n{query}\n\n'
        f'【回答】\n'
    )
    return prompt, context


# テスト
query = 'ドロップアウトとは何ですか？過学習とどう関係しますか？'
prompt, context = rag_answer(query, vectorstore, top_k=TOP_K)

print(f'質問: {query}')
print(f'\n--- 検索されたコンテキスト（先頭300文字）---')
print(context[:300])
print(f'\n--- 構築されたプロンプト（先頭400文字）---')
print(prompt[:400])

---

## Step 6: チャンクサイズによる比較実験

256 / 512 / 1024 の 3 パターンで同じ質問に検索し、取得されるチャンクを比較します。

In [ ]:
# 実行時間: 約2〜3分
comparison_query = 'LoRA でファインチューニングするとき何が学習されますか？'
results_by_chunk = {}

for chunk_size in [256, 512, 1024]:
    splitter_exp = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=50, length_function=len,
    )
    chunks_exp = splitter_exp.split_documents(documents)
    vs_exp = Chroma.from_documents(
        documents=chunks_exp,
        embedding=embeddings,
        collection_name=f'exp_chunk{chunk_size}',
    )
    retrieved = vs_exp.similarity_search(comparison_query, k=2)
    results_by_chunk[chunk_size] = retrieved

print(f'質問: {comparison_query}\n')
for chunk_size, docs in results_by_chunk.items():
    print(f'=== chunk_size={chunk_size} ===')
    for d in docs:
        print(f'  {d.page_content[:150].strip()}...')
    print()

---

## まとめ

1. **チャンキング** がRAGの精度を大きく左右する。小さすぎると文脈が切れ、大きすぎるとノイズが増える
2. **top-k** を増やすと関連情報が取りやすくなるが、LLM のコンテキスト長を消費する
3. **overlap** を設定することでチャンク境界での情報欠落を緩和できる
4. RAG は知識の「更新頻度が高い」タスクで SFT より優れる

次のハンズオンでは、「素のLLM」「SFT後」「RAGあり」の 3 条件で同一質問セットを評価します。